# EGS Collab Exp2 AMU 34 m worked example

This notebook is a readable companion to `scripts/reproduce_egs_collab_exp2.py`. The script is still the canonical reproduction path; this notebook shows the evidence ladder for a technical reviewer.

Evidence ladder: processed public-source-derived rows → thermal recovery fit → boundary conversion → published `m²` scale check → claim boundary.


## 1. Load the small processed inputs

The raw upstream public datasets are not stored in this repository. The included CSVs are small processed snippets used only to reproduce this worked example.


In [ ]:
from pathlib import Path
import csv, json, math
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'processed'

def read_rows(path):
    with path.open(newline='') as handle:
        return list(csv.DictReader(handle))

recovery_rows = read_rows(DATA / 'egs_collab_exp2_amu34m_recovery.csv')
boundary_rows = read_rows(DATA / 'egs_collab_exp2_boundary_inputs.csv')
comparison_rows = read_rows(DATA / 'egs_collab_exp2_published_k_ranges.csv')

elapsed_h = np.array([float(r['elapsed_h']) for r in recovery_rows])
delta_c = np.array([float(r['temperature_delta_c']) for r in recovery_rows])
boundary = {r['name']: float(r['value']) for r in boundary_rows}

print(f'recovery rows: {len(recovery_rows)}')
print(f'elapsed window: {elapsed_h.min():.2f}–{elapsed_h.max():.2f} h after pump stop')
print(f'temperature anomaly range: {delta_c.min():.3f} to {delta_c.max():.3f} °C')
print(f'flow used: {boundary["flow_l_min_pre_stop_median"]:.3f} L/min')
print(f'deltaP used: {boundary["delta_pressure_mpa"]:.1f} MPa')


## 2. Fit a recovery descriptor

`tau` is the fitted recovery timescale. It is not permeability by itself; it is a compact description of how fast the temperature anomaly relaxes.


In [ ]:
def fit_exponential(elapsed_h, delta_c):
    def solve(tau_h):
        shape = np.exp(-elapsed_h / tau_h)
        design = np.column_stack([np.ones_like(shape), shape])
        baseline, amplitude = np.linalg.lstsq(design, delta_c, rcond=None)[0]
        fitted = baseline + amplitude * shape
        rmse = float(np.sqrt(np.mean((delta_c - fitted) ** 2)))
        return float(baseline), float(amplitude), rmse, fitted

    low, high = 0.2, 40.0
    phi = (1 + math.sqrt(5)) / 2
    for _ in range(90):
        left = high - (high - low) / phi
        right = low + (high - low) / phi
        if solve(left)[2] < solve(right)[2]:
            high = right
        else:
            low = left
    tau_h = (low + high) / 2
    baseline, amplitude, rmse, fitted = solve(tau_h)
    return tau_h, baseline, amplitude, rmse, fitted

tau_h, baseline_c, amplitude_c, rmse_c, fitted = fit_exponential(elapsed_h, delta_c)
print(f'tau = {tau_h:.2f} h')
print(f'RMSE = {rmse_c:.4f} °C')

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.scatter(elapsed_h, delta_c, s=24, color='#2563eb', alpha=0.78, label='Measured recovery')
ax.plot(elapsed_h, fitted, color='#dc2626', linewidth=2.8, label=f'Exponential descriptor: tau = {tau_h:.2f} h')
ax.axhline(0, color='#475569', linewidth=1, alpha=0.45)
ax.set_xlabel('Hours after pump stop')
ax.set_ylabel('Temperature anomaly (°C)')
ax.set_title('Raw thermal recovery and descriptor fit')
ax.grid(True, color='#e2e8f0')
ax.legend()
plt.show()


## 3. Convert the boundary calculation to `m²`

The permeability number uses pressure, flow, viscosity, path length, and contact area. This is why the result is called **bulk/model-equivalent permeability**, not measured rock permeability.


In [ ]:
def calc_k(length_m, area_m2):
    flow_m3_s = boundary['flow_l_min_pre_stop_median'] / 1000 / 60
    delta_pressure_pa = boundary['delta_pressure_mpa'] * 1_000_000
    mu = boundary['water_viscosity']
    return mu * flow_m3_s * length_m / (area_m2 * delta_pressure_pa)

k_preferred = calc_k(boundary['path_length_preferred'], boundary['contact_area_preferred'])
k_low = calc_k(boundary['path_length_low'], boundary['contact_area_high'])
k_high = calc_k(boundary['path_length_high'], boundary['contact_area_low'])

print(f'preferred k_eq = {k_preferred:.2e} m²')
print(f'source-constrained band = {k_low:.2e}–{k_high:.2e} m²')


## 4. Compare with published scale ranges

The comparison is same-testbed and same-depth-scale. It is not exact same-fracture or exact same-sample validation.


In [ ]:
labels = ['This worked example']
lows = [k_low]
highs = [k_high]
centers = [k_preferred]
colors = ['#2563eb']

for row in comparison_rows:
    if row['category'] == 'our model-equivalent result':
        continue
    labels.append(row['label'].split(' — ')[0])
    lows.append(float(row['min_m2']))
    highs.append(float(row['max_m2']))
    centers.append(float('nan') if row['central_m2'] == '' else float(row['central_m2']))
    colors.append('#16a34a')

y = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(8, 4.8))
for idx, (low, high, center, color) in enumerate(zip(lows, highs, centers, colors)):
    ax.plot([low, high], [idx, idx], color=color, linewidth=8, solid_capstyle='round')
    if math.isfinite(center):
        ax.scatter([center], [idx], s=110, color='#111827', zorder=3)
ax.set_xscale('log')
ax.set_xlim(1e-18, 1e-12)
ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel('Permeability (m²)')
ax.set_title('Bulk/model-equivalent permeability scale check')
ax.grid(True, axis='x', which='both', color='#e2e8f0')
plt.show()


## Claim boundary

Allowed: one public EGS Collab recovery crop can be reduced to a thermal descriptor and mapped to a bulk/model-equivalent `m²` scale under declared pressure/flow/geometry assumptions.

Not allowed: measured rock permeability, exact-fracture permeability, exact same-sample validation, geometry-independent permeability, or validation of every candidate dataset.
